<h1> Importing Libraries and Loading Datasets </h1>

In [ ]:
import pandas as pd

In [ ]:
books_df = pd.read_json("goodreads_books_children.json", lines = True)

In [ ]:
reviews_df = pd.read_json("goodreads_reviews_children.json", lines = True)

In [ ]:
interactions_df = pd.read_json("goodreads_interactions_children.json", lines = True)

<h1> High-Level Dataset Overview </h1>

In [ ]:
# Helper functions for EDA
import pandas as pd
import numpy as np
from IPython.display import display, HTML

def get_dataset_overview(df, name="Dataset"):
    """
    Generate a comprehensive overview of a dataset.

    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to analyze
    name : str
        Name of the dataset for display

    Returns:
    --------
    dict : Dictionary containing all EDA metrics
    """
    print(f"\n{'='*60}")
    print(f"DATASET: {name}")
    print(f"{'='*60}\n")

    # 1. Shape information
    n_rows, n_cols = df.shape
    print(f"📊 SHAPE INFORMATION:")
    print(f"   Rows: {n_rows:,}")
    print(f"   Columns: {n_cols}")

    # 2. Data types
    print(f"\n📋 DATA TYPES:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"   {dtype}: {count}")

    # 3. Missing values analysis
    print(f"\n❌ MISSING VALUES:")
    missing_data = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum().values,
        'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2).values
    })
    missing_data = missing_data.sort_values('Missing_Count', ascending=False)

    # Display only columns with missing values
    if missing_data['Missing_Count'].sum() > 0:
        print(missing_data[missing_data['Missing_Count'] > 0].to_string(index=False))
    else:
        print("   No missing values found! ✅")

    # 4. Duplicate rows (only check hashable columns)
    try:
        # Select only hashable columns (exclude list, dict, etc.)
        hashable_cols = []
        for col in df.columns:
            try:
                # Test if column is hashable by trying to get unique values
                hash(tuple(df[col].iloc[0] if not pd.isna(df[col].iloc[0]) else None for _ in [0]))
                hashable_cols.append(col)
            except TypeError:
                pass

        if hashable_cols:
            n_duplicates = df[hashable_cols].duplicated().sum()
        else:
            n_duplicates = 0
    except:
        n_duplicates = 0

    print(f"\n🔄 DUPLICATES:")
    print(f"   Duplicate rows: {n_duplicates}")
    print(f"   Duplicate percentage: {(n_duplicates/len(df)*100):.2f}%")

    # 5. Summary statistics for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\n📈 SUMMARY STATISTICS (Numeric Columns):")
        print(df[numeric_cols].describe().round(2).to_string())
    else:
        print(f"\n📈 SUMMARY STATISTICS: No numeric columns found")

    # Return summary as dictionary
    summary = {
        'name': name,
        'rows': n_rows,
        'columns': n_cols,
        'data_types': df.dtypes.to_dict(),
        'missing_values': missing_data,
        'duplicates': n_duplicates,
        'numeric_columns': list(numeric_cols)
    }

    return summary

def compare_datasets_summary(summaries):
    """
    Create a comparison table of dataset summaries.

    Parameters:
    -----------
    summaries : list of dict
        List of summary dictionaries from get_dataset_overview()
    """
    print(f"\n\n{'='*60}")
    print(f"COMPARISON TABLE")
    print(f"{'='*60}\n")

    comparison_data = {
        'Dataset': [s['name'] for s in summaries],
        'Rows': [s['rows'] for s in summaries],
        'Columns': [s['columns'] for s in summaries],
        'Duplicate Rows': [s['duplicates'] for s in summaries],
        'Numeric Cols': [len(s['numeric_columns']) for s in summaries],
        'Has Missing Values': [s['missing_values']['Missing_Count'].sum() > 0 for s in summaries]
    }

    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))

    return comparison_df

In [ ]:
# Analyze Books Dataset
books_summary = get_dataset_overview(books_df, name="Books")


DATASET: Books

📊 SHAPE INFORMATION:
   Rows: 124,082
   Columns: 29

📋 DATA TYPES:
   object: 24
   int64: 4
   float64: 1

❌ MISSING VALUES:
   No missing values found! ✅

🔄 DUPLICATES:
   Duplicate rows: 0
   Duplicate percentage: 0.00%

📈 SUMMARY STATISTICS (Numeric Columns):
       text_reviews_count  average_rating      book_id  ratings_count      work_id
count           124082.00       124082.00    124082.00      124082.00    124082.00
mean                27.09            3.91  10579293.30         522.82  12808871.30
std                266.55            0.36  10179903.74       10838.69  16548772.89
min                  0.00            0.00         5.00           0.00        81.00
25%                  2.00            3.71   1414649.50          10.00   1131108.00
50%                  5.00            3.94   7068956.00          30.00   3022627.00
75%                 15.00            4.14  18165059.25          96.00  20497562.00
max              49850.00            5.00  36469877.00

/var/folders/8g/hms_xq0n2kn06w8md2s7nppw0000gn/T/ipykernel_1446/2970725304.py:59: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  hash(tuple(df[col].iloc[0] if not pd.isna(df[col].iloc[0]) else None for _ in [0]))


In [ ]:
# Analyze Interactions Dataset
interactions_summary = get_dataset_overview(interactions_df, name="Interactions")


DATASET: Interactions

📊 SHAPE INFORMATION:
   Rows: 10,059,349
   Columns: 10

📋 DATA TYPES:
   object: 7
   int64: 2
   bool: 1

❌ MISSING VALUES:
   No missing values found! ✅

🔄 DUPLICATES:
   Duplicate rows: 0
   Duplicate percentage: 0.00%

📈 SUMMARY STATISTICS (Numeric Columns):
           book_id       rating
count  10059349.00  10059349.00
mean    5066247.20         2.55
std     8515882.13         2.08
min           5.00         0.00
25%       65119.00         0.00
50%      430461.00         3.00
75%     6798777.00         4.00
max    36469877.00         5.00


In [ ]:
# Analyze Reviews Dataset
reviews_summary = get_dataset_overview(reviews_df, name="Reviews")


DATASET: Reviews

📊 SHAPE INFORMATION:
   Rows: 734,640
   Columns: 11

📋 DATA TYPES:
   object: 7
   int64: 4

❌ MISSING VALUES:
   No missing values found! ✅

🔄 DUPLICATES:
   Duplicate rows: 0
   Duplicate percentage: 0.00%

📈 SUMMARY STATISTICS (Numeric Columns):
           book_id     rating    n_votes  n_comments
count    734640.00  734640.00  734640.00   734640.00
mean    9520406.08       3.82       0.55        0.14
std    10028764.51       1.24       5.49        1.69
min           5.00       0.00      -3.00       -1.00
25%      432463.00       3.00       0.00        0.00
50%     6411331.00       4.00       0.00        0.00
75%    17349203.00       5.00       0.00        0.00
max    36469877.00       5.00    2764.00      930.00


In [ ]:
# Create comparison table across all datasets
comparison_table = compare_datasets_summary([books_summary, interactions_summary, reviews_summary])



COMPARISON TABLE

     Dataset     Rows  Columns  Duplicate Rows  Numeric Cols  Has Missing Values
       Books   124082       29               0             5               False
Interactions 10059349       10               0             2               False
     Reviews   734640       11               0             4               False


<h1> Pseudo-Missing Values Detection </h1>

In [ ]:
# Pseudo-Missing Values Detection

def detect_pseudo_missing_in_strings(df):
    """
    Detect pseudo-missing values in object/string columns.

    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to analyze

    Returns:
    --------
    pd.DataFrame : Summary table with pseudo-missing counts and percentages
    """
    string_cols = df.select_dtypes(include=['object']).columns

    results = []

    for col in string_cols:
        # Count different types of pseudo-missing values
        null_count = df[col].isnull().sum()
        empty_str = (df[col].astype(str).str.strip() == '').sum()

        # Common placeholder patterns (case-insensitive)
        placeholders = ['none', 'null', 'n/a', 'nan', '[]', '{}']
        placeholder_counts = {}

        for placeholder in placeholders:
            count = (df[col].astype(str).str.lower().str.strip() == placeholder).sum()
            placeholder_counts[f"'{placeholder}'"] = count

        # Total pseudo-missing
        total_pseudo_missing = null_count + empty_str + sum(placeholder_counts.values())
        pct = (total_pseudo_missing / len(df) * 100).round(2)

        results.append({
            'Column': col,
            'Null': null_count,
            'Empty_String': empty_str,
            **placeholder_counts,
            'Total_Pseudo_Missing': total_pseudo_missing,
            'Percentage': pct
        })

    result_df = pd.DataFrame(results)
    return result_df.sort_values('Total_Pseudo_Missing', ascending=False)

def detect_sentinel_values_numeric(df, numeric_cols=None):
    """
    Detect potential sentinel values in numeric columns using frequency analysis.
    Focus on columns like 'rating' where 0 might mean "missing".

    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to analyze
    numeric_cols : list, optional
        Specific numeric columns to analyze. If None, analyzes numeric columns with 'rating' in name.

    Returns:
    --------
    dict : Dictionary of {column: frequency table}
    """
    if numeric_cols is None:
        numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                       if 'rating' in col.lower() or col.lower() in ['rating', 'score', 'value']]

    sentinel_results = {}
    for col in numeric_cols:
        if col in df.columns:
            freq_table = df[col].value_counts().sort_index().head(15)
            sentinel_results[col] = freq_table

    return sentinel_results

def detect_empty_before_datetime(df):
    """
    Check for empty strings before datetime parsing attempts on object columns
    that might be timestamps.

    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to analyze

    Returns:
    --------
    pd.DataFrame : Summary of potential datetime columns with empty string counts
    """
    string_cols = df.select_dtypes(include=['object']).columns

    results = []
    for col in string_cols:
        # Check for common datetime column name patterns
        if any(keyword in col.lower() for keyword in ['date', 'time', 'created', 'updated', 'published', 'timestamp']):
            empty_count = (df[col].astype(str).str.strip() == '').sum()
            null_count = df[col].isnull().sum()

            results.append({
                'Column': col,
                'Null_Count': null_count,
                'Empty_String_Count': empty_count,
                'Total_Empty': null_count + empty_count,
                'Percentage': ((null_count + empty_count) / len(df) * 100).round(2)
            })

    if results:
        return pd.DataFrame(results).sort_values('Total_Empty', ascending=False)
    else:
        return pd.DataFrame()

def print_sentinel_values(sentinel_dict, col_name):
    """Helper function to pretty print sentinel value frequencies"""
    print(f"\n{'─'*60}")
    print(f"Column: {col_name}")
    print(f"{'─'*60}")
    if col_name in sentinel_dict:
        freq_table = sentinel_dict[col_name]
        for value, count in freq_table.items():
            pct = (count / freq_table.sum() * 100).round(2)
            print(f"  Value {value}: {count:,} ({pct}%)")
    else:
        print(f"  No data for {col_name}")

print("✅ Pseudo-missing value detection functions loaded")

✅ Pseudo-missing value detection functions loaded


In [ ]:
# Analyze Books Dataset - Pseudo-Missing Values

print("\n" + "="*70)
print("BOOKS DATASET - PSEUDO-MISSING VALUES ANALYSIS")
print("="*70)

# String columns with pseudo-missing values
books_pseudo_strings = detect_pseudo_missing_in_strings(books_df)
print(f"\n📝 STRING COLUMNS - Pseudo-Missing Values:")
print(books_pseudo_strings.to_string(index=False))

# Sentinel values in numeric columns (especially ratings)
books_sentinels = detect_sentinel_values_numeric(books_df)
print(f"\n📊 NUMERIC COLUMNS - Potential Sentinel Values (Frequency):")
for col in books_sentinels.keys():
    print_sentinel_values(books_sentinels, col)

# Datetime columns
books_datetime = detect_empty_before_datetime(books_df)
if not books_datetime.empty:
    print(f"\n📅 DATETIME COLUMNS - Empty String Counts:")
    print(books_datetime.to_string(index=False))
else:
    print(f"\n📅 DATETIME COLUMNS: No potential datetime columns detected")


BOOKS DATASET - PSEUDO-MISSING VALUES ANALYSIS

📝 STRING COLUMNS - Pseudo-Missing Values:
              Column  Null  Empty_String  'none'  'null'  'n/a'  'nan'  '[]'  '{}'  Total_Pseudo_Missing  Percentage
 edition_information     0        118307       0       0      0      0     0     0                118307       95.35
                asin     0        117363       0       0      0      0     0     0                117363       94.59
         kindle_asin     0         82960       0       0      0      0     0     0                 82960       66.86
              series     0             0       0       0      0      0 81213     0                 81213       65.45
       language_code     0         72496       0       0      0      0     0     0                 72496       58.43
       similar_books     0             0       0       0      0      0 65774     0                 65774       53.01
     publication_day     0         38472       0       0      0      0     0     0        

In [ ]:
# Analyze Interactions Dataset - Pseudo-Missing Values

print("\n" + "="*70)
print("INTERACTIONS DATASET - PSEUDO-MISSING VALUES ANALYSIS")
print("="*70)

# String columns with pseudo-missing values
interactions_pseudo_strings = detect_pseudo_missing_in_strings(interactions_df)
print(f"\n📝 STRING COLUMNS - Pseudo-Missing Values:")
print(interactions_pseudo_strings.to_string(index=False))

# Sentinel values in numeric columns (especially ratings)
interactions_sentinels = detect_sentinel_values_numeric(interactions_df)
print(f"\n📊 NUMERIC COLUMNS - Potential Sentinel Values (Frequency):")
if interactions_sentinels:
    for col in interactions_sentinels.keys():
        print_sentinel_values(interactions_sentinels, col)
else:
    print("  No rating/score columns found")

# Datetime columns
interactions_datetime = detect_empty_before_datetime(interactions_df)
if not interactions_datetime.empty:
    print(f"\n📅 DATETIME COLUMNS - Empty String Counts:")
    print(interactions_datetime.to_string(index=False))
else:
    print(f"\n📅 DATETIME COLUMNS: No potential datetime columns detected")


INTERACTIONS DATASET - PSEUDO-MISSING VALUES ANALYSIS

📝 STRING COLUMNS - Pseudo-Missing Values:
                Column  Null  Empty_String  'none'  'null'  'n/a'  'nan'  '[]'  '{}'  Total_Pseudo_Missing  Percentage
review_text_incomplete     0       9322667       0       3      1      0     0     0               9322671       92.68
            started_at     0       9071761       0       0      0      0     0     0               9071761       90.18
               read_at     0       8174589       0       0      0      0     0     0               8174589       81.26
               user_id     0             0       0       0      0      0     0     0                     0        0.00
             review_id     0             0       0       0      0      0     0     0                     0        0.00
            date_added     0             0       0       0      0      0     0     0                     0        0.00
          date_updated     0             0       0       0      0    

In [ ]:
# Analyze Reviews Dataset - Pseudo-Missing Values

print("\n" + "="*70)
print("REVIEWS DATASET - PSEUDO-MISSING VALUES ANALYSIS")
print("="*70)

# String columns with pseudo-missing values
reviews_pseudo_strings = detect_pseudo_missing_in_strings(reviews_df)
print(f"\n📝 STRING COLUMNS - Pseudo-Missing Values:")
print(reviews_pseudo_strings.to_string(index=False))

# Sentinel values in numeric columns (especially ratings)
reviews_sentinels = detect_sentinel_values_numeric(reviews_df)
print(f"\n📊 NUMERIC COLUMNS - Potential Sentinel Values (Frequency):")
if reviews_sentinels:
    for col in reviews_sentinels.keys():
        print_sentinel_values(reviews_sentinels, col)
else:
    print("  No rating/score columns found")

# Datetime columns
reviews_datetime = detect_empty_before_datetime(reviews_df)
if not reviews_datetime.empty:
    print(f"\n📅 DATETIME COLUMNS - Empty String Counts:")
    print(reviews_datetime.to_string(index=False))
else:
    print(f"\n📅 DATETIME COLUMNS: No potential datetime columns detected")


REVIEWS DATASET - PSEUDO-MISSING VALUES ANALYSIS

📝 STRING COLUMNS - Pseudo-Missing Values:
      Column  Null  Empty_String  'none'  'null'  'n/a'  'nan'  '[]'  '{}'  Total_Pseudo_Missing  Percentage
  started_at     0        466008       0       0      0      0     0     0                466008       63.43
     read_at     0        180180       0       0      0      0     0     0                180180       24.53
 review_text     0           158       0       3      1      0     0     0                   162        0.02
     user_id     0             0       0       0      0      0     0     0                     0        0.00
   review_id     0             0       0       0      0      0     0     0                     0        0.00
  date_added     0             0       0       0      0      0     0     0                     0        0.00
date_updated     0             0       0       0      0      0     0     0                     0        0.00

📊 NUMERIC COLUMNS - Potential Sent

In [ ]:
# Summary - Pseudo-Missing Values Overview

print("\n" + "="*70)
print("PSEUDO-MISSING VALUES SUMMARY ACROSS ALL DATASETS")
print("="*70)

# Summary of string columns with worst pseudo-missing rates
print("\n🔴 TOP COLUMNS WITH PSEUDO-MISSING VALUES (%):\n")

all_pseudo_data = []
for name, df, pseudo_df in [("Books", books_df, books_pseudo_strings),
                             ("Interactions", interactions_df, interactions_pseudo_strings),
                             ("Reviews", reviews_df, reviews_pseudo_strings)]:
    top = pseudo_df.nlargest(3, 'Total_Pseudo_Missing')[['Column', 'Total_Pseudo_Missing', 'Percentage']]
    for _, row in top.iterrows():
        all_pseudo_data.append({
            'Dataset': name,
            'Column': row['Column'],
            'Pseudo_Missing_Count': row['Total_Pseudo_Missing'],
            'Percentage': row['Percentage']
        })

summary_pseudo_df = pd.DataFrame(all_pseudo_data)
if not summary_pseudo_df.empty:
    print(summary_pseudo_df.sort_values('Percentage', ascending=False).to_string(index=False))
else:
    print("No pseudo-missing values detected in string columns")

# Summary of sentinel values in rating columns
print("\n\n⚠️  KEY SENTINEL VALUE FINDINGS:\n")

sentinel_findings = [
    ("Books", books_sentinels, "average_rating"),
    ("Interactions", interactions_sentinels, "rating"),
    ("Reviews", reviews_sentinels, "rating")
]

for dataset_name, sentinels, col_name in sentinel_findings:
    if col_name in sentinels:
        freq = sentinels[col_name]
        zero_count = freq.get(0, 0) if 0 in freq.index else 0
        zero_pct = (zero_count / freq.sum() * 100).round(2) if zero_count > 0 else 0

        print(f"{dataset_name} - '{col_name}':")
        print(f"  • Value 0: {zero_count:,} ({zero_pct}%) - potential 'missing' sentinel")
        print(f"  • Value 5: {freq.get(5, 0):,} ({(freq.get(5, 0)/freq.sum()*100).round(2)}%)")
        print(f"  • Value 4: {freq.get(4, 0):,} ({(freq.get(4, 0)/freq.sum()*100).round(2)}%)")
        print()
    else:
        print(f"{dataset_name}: No rating column found\n")

print("\n✅ Analysis complete. Check output cells above for detailed frequency tables.")


PSEUDO-MISSING VALUES SUMMARY ACROSS ALL DATASETS

🔴 TOP COLUMNS WITH PSEUDO-MISSING VALUES (%):

     Dataset                 Column  Pseudo_Missing_Count  Percentage
       Books    edition_information                118307       95.35
       Books                   asin                117363       94.59
Interactions review_text_incomplete               9322671       92.68
Interactions             started_at               9071761       90.18
Interactions                read_at               8174589       81.26
       Books            kindle_asin                 82960       66.86
     Reviews             started_at                466008       63.43
     Reviews                read_at                180180       24.53
     Reviews            review_text                   162        0.02


⚠️  KEY SENTINEL VALUE FINDINGS:

Books - 'average_rating':
  • Value 0: 23 (36.51%) - potential 'missing' sentinel
  • Value 5: 0 (0.0%)
  • Value 4: 0 (0.0%)

Interactions - 'rating':
  • Value 0: 

From EDA, String Columns with High Pseudo-Missing Rates
- edition_information (books_df)
- asin (books_df)
- kindle_asin (books_df)
- review_text_incomplete (interactions_df)
- started_at (interactions_df)
- read_at (interactions_df)
- started_at (reviews_df)
- read_at (reviews_df)

Hence, we opt to remove edition_information, asin, kindle_asin for modelling.

While started_at and read_at are sparse, they are still useful to help distinguish reading progress/ completion patterns, and can help to define is_read logic checks.